## PROYECTO : detectar toxicidad en comentarios online

La detección de toxicidad permite **identificar lenguaje ofensivo, insultos o discursos de odio** en redes sociales, foros o comentarios.

Es fundamental para:

- 🛡️ **Proteger comunidades online** (YouTube, Wikipedia, X, Reddit).  
- ⚙️ **Filtrar contenido automáticamente.**  
- 🤖 **Entrenar moderadores automáticos con IA.**

## Exploración del dataset Jigsaw Toxic Comment Classification

In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("jhan21/jigsaw-toxic-comment-classification", split="train[:2%]")
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.2)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.csv:   0%|          | 0.00/68.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/159571 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 2552
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 639
    })
})

## convertimos el dataset a DataFrames de pandas

In [2]:
import pandas as pd

# Convert the train dataset to a pandas DataFrame for easier analysis
train_df = dataset["train"].to_pandas()

# Get the category columns
category_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Calculate the number of comments per category
category_counts = train_df[category_cols].sum()

print("Número de comentarios por categoría en el conjunto de entrenamiento:")
print(category_counts)

Número de comentarios por categoría en el conjunto de entrenamiento:
toxic            273
severe_toxic      30
obscene          143
threat            14
insult           151
identity_hate     24
dtype: int64


In [3]:
train_df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00c381d0018beaf3,Btw whatever decision an adminstration wish to...,0,0,0,0,0,0
1,009e3f1c0c757e43,"""\nIt's in the History, or more conveniently i...",0,0,0,0,0,0
2,0091798f05a311af,http://www.users.bigpond.com/MONTDALE/page8.ht...,0,0,0,0,0,0
3,050b805625699bb5,"""\n\nMidwest Accent\nI see that you are deleti...",0,0,0,0,0,0
4,06751fbb70781e52,"""\nOppose – As pointed above, the other articl...",0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
2547,04244108a528db67,"Elvek, on October 3, 2010\nThanks very much JR...",0,0,0,0,0,0
2548,072825b5cedee27f,The Greater San Francisco Bay Area is greater ...,1,0,0,0,1,0
2549,06d66b115a738039,"""\n\n Please do not vandalize pages, as you di...",0,0,0,0,0,0
2550,06700c6a81f2dce6,"""\nI'm being blocked for comments that I made ...",0,0,0,0,0,0


# Configuración y Entrenamiento del Modelo BERT

## 📘 Carga del modelo preentrenado

In [4]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model_name = "distilbert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(model_name)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 📘 Preparación de DataLoader y pipeline de entrenamiento

In [5]:
def tokenize(batch):
    return tokenizer(batch["comment_text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("toxic", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Map:   0%|          | 0/2552 [00:00<?, ? examples/s]

Map:   0%|          | 0/639 [00:00<?, ? examples/s]

## Entrenamiento de Trainer con Hugging Face

In [7]:
from transformers import TrainingArguments, Trainer

new_model_name = "cesarcodigo-toxicidad-g6"
# Configurar entrenamiento
training_args = TrainingArguments(
    output_dir=f"./{new_model_name}",          # Directorio de salida corregido
    num_train_epochs=3,                       # Número total de épocas de entrenamiento
    per_device_train_batch_size=16,           # Tamaño del batch por dispositivo durante el entrenamiento
    per_device_eval_batch_size=64,            # Tamaño del batch para evaluación
    warmup_steps=500,                         # Número de pasos de calentamiento para el scheduler de tasa de aprendizaje
    weight_decay=0.01,                        # Fuerza de la penalización L2
    logging_dir=f"./logs_{new_model_name}",   # Directorio para almacenar logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,0.000085
20,0.000072
30,0.000074
40,0.000061
50,0.000139
60,0.030966
70,0.000033
80,0.000034
90,0.000034
100,0.000026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=480, training_loss=0.0006622442623760587, metrics={'train_runtime': 118.0228, 'train_samples_per_second': 64.869, 'train_steps_per_second': 4.067, 'total_flos': 253542601027584.0, 'train_loss': 0.0006622442623760587, 'epoch': 3.0})

## Prueba del modelo con un texto de ejemplo

In [8]:
import torch

# Texto de ejemplo tóxico
toxic_text = "You are an idiot and your comments are stupid!"

# Tokenizar el texto
inputs = tokenizer(toxic_text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

# Mover los inputs al mismo dispositivo que el modelo (GPU si está disponible, CPU en caso contrario)
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Realizar la predicción
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Obtener las probabilidades (logits) y la clase predicha
logits = outputs.logits
probabilities = torch.softmax(logits, dim=1)
predicted_class_id = torch.argmax(logits, dim=1).item()

# Interpretar el resultado (clase 0: no tóxico, clase 1: tóxico)
prediction_label = "Tóxico" if predicted_class_id == 1 else "No Tóxico"

print(f"Texto de entrada: '{toxic_text}'")
print(f"Logits: {logits.cpu().numpy()}")
print(f"Probabilidades: {probabilities.cpu().numpy()}")
print(f"Clase predicha (0=No tóxico, 1=Tóxico): {predicted_class_id}")
print(f"Resultado: {prediction_label}")

Texto de entrada: 'You are an idiot and your comments are stupid!'
Logits: [[-6.2474384  6.6837506]]
Probabilidades: [[2.4213364e-06 9.9999762e-01]]
Clase predicha (0=No tóxico, 1=Tóxico): 1
Resultado: Tóxico


# GUARDAMOS EL MODELO

In [9]:
trainer.save_model(new_model_name)
tokenizer.save_pretrained(new_model_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('cesarcodigo-toxicidad-g6/tokenizer_config.json',
 'cesarcodigo-toxicidad-g6/tokenizer.json')

In [10]:
!hf auth login

A new version of huggingface_hub (1.6.0) is available! You are using version 1.5.0.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: n
Token is valid (permission: write).
The token `codigog6` has been saved to /root/.cache/huggingface/stor

# PUBLICAMOS EL MODELO EN HUGGING FACE

In [11]:
trainer.push_to_hub(new_model_name)

print(f"¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre '{new_model_name}'!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...idad-g6/model.safetensors:   0%|          |  575kB /  268MB            

  ...idad-g6/training_args.bin:   1%|          |  43.0B / 5.20kB            

¡El modelo ha sido re-publicado exitosamente en Hugging Face con el nombre 'cesarcodigo-toxicidad-g6'!
